In [4]:
# Import libraries
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from pathlib import Path
import polars as pl

# analytics_db connection
# Define project paths and load local database credentials from the .env file
PROJECT_PATH = Path(
    r"C:\Users\zychl\Documents\GitHub\data-toolkit\weather_data_processing"
)

load_dotenv(PROJECT_PATH / ".env") 

# PostgreSQL connection details
username = os.getenv('POSTGRES_USER')
password = os.getenv('POSTGRES_PASSWORD')
host = 'localhost'
port = os.getenv('HOST_POSTGRES_PORT')
database = os.getenv('POSTGRES_DB')

engine = create_engine(
    f'postgresql+pg8000://{username}:{password}@{host}:{port}/{database}'
    )

# Fetch the data from PostgreSQL analytics_db server - bronze layer
print('Executing query...')

bronze_max_date = pl.read_database(
    query=
    'SELECT MAX(observation_date) AS max_observation_date ' \
    'FROM bronze.weather',
    connection=engine
    )

bronze_max_date = bronze_max_date['max_observation_date'][0]

print(f'Executing query finished. The result is {bronze_max_date}')

Executing query...
Executing query finished. The result is 2026-07-16


In [ ]:
# ======================================================================
# Get the latest weather data from the NOAA
# ======================================================================
from datetime import datetime

current_year = datetime.now().year

url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/{current_year}.csv.gz"

df = pl.read_csv(
    url,
    has_header=False
    )

# Rename columns 
df = df.rename({
    "column_1": "station",
    "column_2": "observation_date",
    "column_3": "metric",
    "column_4": "value",
    "column_5": "measurement_flag",
    "column_6": "quality_flag",
    "column_7": "source_flag",
    "column_8": "observation_time"
})

In [21]:
# Convert 'observation_date' to YYYY-MM-DD
df = df.with_columns(
    pl.col('observation_date')
    .cast(pl.String)
    .str.strptime(pl.Date, '%Y%m%d')
)

In [23]:
# Extract max date from df 
df_max_date = df.select(
    pl.col('observation_date').max()
    )

df_max_date

observation_date
date
2026-10-13


In [27]:
pl.col("observation_date") > bronze_max_date

<Expr ['[(col("observation_date")) > (…'] at 0x148EB0D6650>